In [3]:
from google.colab import drive
drive.mount('/content/drive')

print("Drive mounted at /content/drive/")
!ls /content/drive


Mounted at /content/drive
Drive mounted at /content/drive/
MyDrive  Othercomputers


In [ ]:
import os, re, numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import defaultdict
import joblib
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TextClassificationPipeline
import torch

BASE_DIR = "/content/drive/MyDrive/longformer_runs/run_paper_v1"

DATA_DIR    = f"{BASE_DIR}/data"
RESULTS_DIR = f"{BASE_DIR}/results_longformer"
SHAP_DIR    = f"{BASE_DIR}/shap_outputs"

print("DATA_DIR   :", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("SHAP_DIR   :", SHAP_DIR)

print("\nListing data directory:")
!ls "$DATA_DIR"

print("\nListing SHAP outputs directory:")
!ls "$SHAP_DIR"

test_json_path = os.path.join(DATA_DIR, "test.json")
if not os.path.isfile(test_json_path):
    raise FileNotFoundError(f"Could not find test.json at {test_json_path}")

test_df = pd.read_json(test_json_path)

def combine_example(code, comment):
    return (code or "") + "\n\n[COMMENT]\n" + (comment or "")

all_texts = [
    combine_example(c, m)
    for c, m in zip(test_df["new_code_raw"], test_df["new_comment_raw"])
]
all_labels = test_df["label"].astype(int).to_numpy()

print("Total test samples:", len(all_texts))

pkl_path = os.path.join(SHAP_DIR, "shap_values_batched.pkl")
if not os.path.isfile(pkl_path):
    raise FileNotFoundError(f"Missing shap_values_batched.pkl at {pkl_path}")

shap_values, sample_texts = joblib.load(pkl_path)
print("Loaded SHAP values successfully.")

subset_path = os.path.join(SHAP_DIR, "shap_samples_batched_with_row.csv")
shap_samples_df = pd.read_csv(subset_path)
print("Loaded SHAP subset:", subset_path)


DATA_DIR   : /content/drive/MyDrive/longformer_runs/run_paper_v1/data
RESULTS_DIR: /content/drive/MyDrive/longformer_runs/run_paper_v1/results_longformer
SHAP_DIR   : /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs

Listing data directory:
test.json  valid.json

Listing SHAP outputs directory:
force_plot_batched_0.html  force_plot_batched_7.html
force_plot_batched_1.html  force_plot_batched_8.html
force_plot_batched_2.html  force_plot_batched_9.html
force_plot_batched_3.html  global_top_tokens_class1.png
force_plot_batched_4.html  shap_samples_batched.csv
force_plot_batched_5.html  shap_samples_batched_with_row.csv
force_plot_batched_6.html  shap_values_batched.pkl
Total test samples: 1066
Loaded SHAP values successfully.
Loaded SHAP subset: /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs/shap_samples_batched_with_row.csv


In [ ]:
best_ckpt_file = os.path.join(RESULTS_DIR, "BEST_CHECKPOINT.txt")
if not os.path.isfile(best_ckpt_file):
    raise FileNotFoundError("BEST_CHECKPOINT.txt not found!")

with open(best_ckpt_file) as f:
    best_ckpt = f.read().strip()

print("Using checkpoint:", best_ckpt)

tokenizer = AutoTokenizer.from_pretrained("allenai/longformer-base-4096")
model = AutoModelForSequenceClassification.from_pretrained(best_ckpt)

device = 0 if torch.cuda.is_available() else -1
clf = TextClassificationPipeline(
    model=model,
    tokenizer=tokenizer,
    device=device,
    return_all_scores=True,
)

pred_csv_path = os.path.join(SHAP_DIR, "test_predictions.csv")

if os.path.isfile(pred_csv_path):
    print("Loading cached predictions...")
    preds_df = pd.read_csv(pred_csv_path)
else:
    print("Computing predictions on full test set...")
    all_probs = []
    all_pred_labels = []
    BATCH = 16

    for start in range(0, len(all_texts), BATCH):
        batch = all_texts[start:start+BATCH]
        outs = clf(batch, truncation=True, max_length=1024)

        for row in outs:
            p0 = p1 = 0.0
            for d in row:
                lab = d["label"]
                idx = int(lab.replace("LABEL_", "")) if lab.startswith("LABEL_") else int(lab)
                if idx == 0:
                    p0 = d["score"]
                else:
                    p1 = d["score"]
            all_probs.append((p0, p1))
            all_pred_labels.append(1 if p1 >= p0 else 0)

    probs_arr = np.array(all_probs)
    preds_df = pd.DataFrame({
        "idx_in_test_after_cleaning": np.arange(len(all_texts)),
        "label_test": all_labels,
        "p_class0": probs_arr[:, 0],
        "p_class1": probs_arr[:, 1],
        "pred_label": all_pred_labels
    })
    preds_df.to_csv(pred_csv_path, index=False)
    print("Saved predictions to:", pred_csv_path)

join_df = shap_samples_df.merge(preds_df, on="idx_in_test_after_cleaning", how="inner")

print("Joined SHAP subset shape:", join_df.shape)
join_df.head()


Using checkpoint: /content/drive/MyDrive/longformer_runs/run_paper_v1/results_longformer/checkpoint-1575


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Device set to use cuda:0
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(
Initializing global attention on CLS token...
Input ids are automatically padded to be a multiple of `config.attention_window`: 512


Computing predictions on full test set...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Saved predictions to: /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs/test_predictions.csv
Joined SHAP subset shape: (30, 8)


,idx_in_test_after_cleaning,text,label,shap_row,label_test,p_class0,p_class1,pred_label
0,295,public ArrayList<ErrorMsg> getWarnings() {...,1,0,1,0.964182,0.035818,0
1,298,public int atAdPos(final int pos) {\n ...,0,1,0,0.901602,0.098398,0
2,361,public List getAnchorHRefs(boolean duplica...,1,2,1,0.931769,0.068231,0
3,869,\tpublic Index parseAndUpdateIndex(List<JaxbRo...,0,3,0,0.921380,0.078620,0
4,915,\tprivate boolean isBreakOnOpcode(Integer opco...,0,4,0,0.762079,0.237921,0


In [ ]:
def clean_token(t):
    if t is None:
        return ""
    s = str(t)
    s = s.replace("Ġ", " ").replace("▁", " ").replace("##", "")
    s = re.sub(r"\s+", " ", s).strip()
    if s in {"", "[CLS]", "[SEP]", "<s>", "</s>", "<pad>"}:
        return ""
    return s

def aggregate_tokens_for_rows(rows, class_idx, top_k=25):
    agg = defaultdict(list)

    for r in rows:
        e = shap_values[r]
        vals = e.values
        toks = e.data

        if vals.ndim == 2:
            vals_class = vals[:, class_idx]
        else:
            vals_class = vals

        for tok, v in zip(toks, vals_class):
            ct = clean_token(tok)
            if ct:
                agg[ct].append(abs(float(v)))

    if not agg:
        return pd.DataFrame(columns=["token", "mean_abs_shap", "count"])

    df = pd.DataFrame({
        "token": list(agg.keys()),
        "mean_abs_shap": [np.mean(v) for v in agg.values()],
        "count": [len(v) for v in agg.values()]
    }).sort_values("mean_abs_shap", ascending=False).head(top_k)

    return df

def plot_bar(df, title, out_path):
    if df.empty:
        print(f"Nothing to plot for {title}")
        return
    plt.figure(figsize=(8, max(4, 0.3*len(df))))
    y = np.arange(len(df))[::-1]
    labels = [f"{t} ({c})" for t, c in zip(df["token"], df["count"])]
    plt.barh(y, df["mean_abs_shap"])
    plt.yticks(y, labels, fontsize=9)
    plt.xlabel("Mean |SHAP|")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()
    print("Saved:", out_path)

# Row groups
tp_rows = join_df[(join_df["label_test"] == 1) & (join_df["pred_label"] == 1)]["shap_row"].tolist()
tn_rows = join_df[(join_df["label_test"] == 0) & (join_df["pred_label"] == 0)]["shap_row"].tolist()
fp_rows = join_df[(join_df["label_test"] == 0) & (join_df["pred_label"] == 1)]["shap_row"].tolist()
fn_rows = join_df[(join_df["label_test"] == 1) & (join_df["pred_label"] == 0)]["shap_row"].tolist()

print("TP:", len(tp_rows), "| TN:", len(tn_rows), "| FP:", len(fp_rows), "| FN:", len(fn_rows))

df_tp = aggregate_tokens_for_rows(tp_rows, 1)
df_tn = aggregate_tokens_for_rows(tn_rows, 0)
df_fp = aggregate_tokens_for_rows(fp_rows, 1)
df_fn = aggregate_tokens_for_rows(fn_rows, 1)

# Save plots
plot_bar(df_tp, "Top Tokens — True Positive (Consistent)", os.path.join(SHAP_DIR, "top_tokens_TP.png"))
plot_bar(df_tn, "Top Tokens — True Negative (Inconsistent)", os.path.join(SHAP_DIR, "top_tokens_TN.png"))
plot_bar(df_fp, "Top Tokens — False Positive (Predicted Consistent)", os.path.join(SHAP_DIR, "top_tokens_FP.png"))
plot_bar(df_fn, "Top Tokens — False Negative (Predicted Inconsistent)", os.path.join(SHAP_DIR, "top_tokens_FN.png"))


TP: 4 | TN: 16 | FP: 0 | FN: 10
Saved: /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs/top_tokens_TP.png
Saved: /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs/top_tokens_TN.png
Nothing to plot for Top Tokens — False Positive (Predicted Consistent)
Saved: /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs/top_tokens_FN.png
